<center>
    
# Portfolio Net Asset Value Tracker 
## **Tools:** Python, yfinance, pandas, openpyxl  
## Anagha Athreyas


## What This Project Is

When you invest money in a fund, the fund manager needs to calculate the value of the fund every single day. This daily value is called the **Net Asset Value** — it tells you exactly how much the fund is worth after accounting for all holdings and costs.

This project automates that entire calculation process from scratch.

---

## Why I Built It

In fund accounting roles at firms like BNY, State Street, and JPMorgan, one of the core daily responsibilities is calculating Net Asset Value for client funds — checking that the numbers are accurate, flagging anything unusual, and producing a report for review.

I built this project to understand and replicate that workflow hands-on, using real market data and institutional logic.

---

## What the Project Does

**Step 1 — Pulls live market prices**  
Uses Yahoo Finance to fetch the latest closing prices for 8 stocks across North America, Europe, and Asia Pacific.

**Step 2 — Calculates Gross Asset Value**  
Adds up the market value of every position (shares multiplied by current price) plus the cash balance held in the fund. This total is the Gross Asset Value — the fund's value before expenses.

**Step 3 — Accrues daily expenses**  
Real funds charge an annual expense ratio (for example, 0.75% per year). Rather than deducting it once a year, it is spread across every trading day. This project calculates and deducts that daily accrual automatically.

**Step 4 — Calculates Net Asset Value and Unit Price**  
Net Asset Value = Gross Asset Value minus the daily expense accrual.  
Unit Price = Net Asset Value divided by the total number of fund units issued.  
This is the price at which investors buy into or redeem from the fund each day.

**Step 5 — Flags exceptions**  
If the Net Asset Value moves more than 2% in a single day, the model flags it for review. In real fund operations, large unexplained moves are escalated to senior staff before the Net Asset Value is published. Flags above 5% are rated HIGH severity.

**Step 6 — Generates an Excel management report**  
Exports a formatted 3-sheet Excel workbook with a full Net Asset Value history, a positions snapshot showing unrealized gains and losses per stock, and an exception log.

---

## What I Learned

- How fund-level Net Asset Value is calculated and why daily expense accrual matters
- How to structure a reconciliation and exception-flagging workflow the way a fund accountant would
- How to automate a reporting process end to end — from live data pull to formatted Excel output — using Python

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'yfinance', 'openpyxl', 'pandas', '--quiet'])

import yfinance as yf
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime, timedelta
import os

print("All libraries loaded.")

All libraries loaded.


In [2]:
PORTFOLIO = {
    "AAPL": {"shares": 50,  "cost_basis": 155.00, "sector": "Technology", "geography": "North America"},
    "MSFT": {"shares": 30,  "cost_basis": 280.00, "sector": "Technology", "geography": "North America"},
    "JPM":  {"shares": 40,  "cost_basis": 145.00, "sector": "Financials", "geography": "North America"},
    "JNJ":  {"shares": 25,  "cost_basis": 160.00, "sector": "Healthcare", "geography": "North America"},
    "XOM":  {"shares": 35,  "cost_basis": 105.00, "sector": "Energy",     "geography": "North America"},
    "ASML": {"shares": 15,  "cost_basis": 620.00, "sector": "Technology", "geography": "Europe"},
    "NVO":  {"shares": 20,  "cost_basis": 100.00, "sector": "Healthcare", "geography": "Europe"},
    "TM":   {"shares": 20,  "cost_basis": 185.00, "sector": "Consumer",   "geography": "Asia Pacific"},
}

TOTAL_UNITS_ISSUED   = 10000
ANNUAL_EXPENSE_RATIO = 0.0075
CASH_BALANCE         = 50000
EXCEPTION_THRESHOLD  = 0.02
LOOKBACK_DAYS        = 30

print(f"Portfolio configured: {len(PORTFOLIO)} positions.")

Portfolio configured: 8 positions.


In [3]:
end_date   = datetime.today()
start_date = end_date - timedelta(days=LOOKBACK_DAYS * 2)
tickers    = list(PORTFOLIO.keys())

print(f"Fetching prices for: {', '.join(tickers)}")
raw = yf.download(tickers, start=start_date.strftime("%Y-%m-%d"),
                  end=end_date.strftime("%Y-%m-%d"), progress=False)["Close"]

if isinstance(raw, pd.Series):
    raw = raw.to_frame(tickers[0])

prices = raw.dropna(how="all").tail(LOOKBACK_DAYS)
print(f"Retrieved {len(prices)} trading days.")
prices.tail(3)

Fetching prices for: AAPL, MSFT, JPM, JNJ, XOM, ASML, NVO, TM
Retrieved 30 trading days.


Ticker,AAPL,ASML,JNJ,JPM,MSFT,NVO,TM,XOM
Date,,,,,,,,
2026-06-08,301.540009,1749.040039,232.160004,311.109985,411.739990,41.020000,178.449997,151.750000
2026-06-09,290.549988,1777.770020,237.000000,312.700012,403.410004,42.189999,175.779999,148.910004
2026-06-10,291.579987,1734.189941,238.490005,309.140015,397.359985,42.810001,172.029999,150.619995


In [4]:
daily_expense_rate = ANNUAL_EXPENSE_RATIO / 252
records = []

for date, row in prices.iterrows():
    gav = sum(
        PORTFOLIO[t]["shares"] * row[t]
        for t in PORTFOLIO
        if t in row.index and pd.notna(row[t])
    ) + CASH_BALANCE

    daily_expense = gav * daily_expense_rate
    nav           = gav - daily_expense
    unit_price    = nav / TOTAL_UNITS_ISSUED

    records.append({
        "Date":                      date,
        "Gross Asset Value ($)":     round(gav, 2),
        "Daily Expense Accrual ($)": round(daily_expense, 2),
        "Net Asset Value ($)":       round(nav, 2),
        "Unit Price ($)":            round(unit_price, 4),
    })

nav_df = pd.DataFrame(records).set_index("Date")
nav_df["Daily NAV Return (%)"] = nav_df["Net Asset Value ($)"].pct_change() * 100
nav_df["Exception Flag"]       = nav_df["Daily NAV Return (%)"].abs() > (EXCEPTION_THRESHOLD * 100)

latest = nav_df.iloc[-1]
print("=" * 45)
print("  NET ASSET VALUE SUMMARY")
print("=" * 45)
print(f"Date:              {nav_df.index[-1].strftime('%Y-%m-%d')}")
print(f"Gross Asset Value: ${latest['Gross Asset Value ($)']:>12,.2f}")
print(f"Net Asset Value:   ${latest['Net Asset Value ($)']:>12,.2f}")
print(f"Unit Price:        ${latest['Unit Price ($)']:>12,.4f}")
print(f"Exception Days:    {int(nav_df['Exception Flag'].sum())} of {len(nav_df)}")
print("=" * 45)
nav_df.tail(5)

  NET ASSET VALUE SUMMARY
Date:              2026-06-10
Gross Asset Value: $  130,409.00
Net Asset Value:   $  130,405.12
Unit Price:        $     13.0405
Exception Days:    0 of 30


,Gross Asset Value ($),Daily Expense Accrual ($),Net Asset Value ($),Unit Price ($),Daily NAV Return (%),Exception Flag
Date,,,,,,
2026-06-04,132691.30,3.95,132687.35,13.2687,0.808919,False
2026-06-05,130456.85,3.88,130452.97,13.0453,-1.683943,False
2026-06-08,131613.85,3.92,131609.93,13.1610,0.886879,False
2026-06-09,131300.60,3.91,131296.69,13.1297,-0.238006,False
2026-06-10,130409.00,3.88,130405.12,13.0405,-0.679050,False


In [5]:
latest_prices = prices.iloc[-1]
positions = []

for ticker, details in PORTFOLIO.items():
    if ticker not in latest_prices.index or pd.isna(latest_prices[ticker]):
        continue
    current_price  = latest_prices[ticker]
    market_value   = details["shares"] * current_price
    cost_value     = details["shares"] * details["cost_basis"]
    unrealized_pnl = market_value - cost_value
    pnl_pct        = (unrealized_pnl / cost_value) * 100

    positions.append({
        "Ticker":                          ticker,
        "Sector":                          details["sector"],
        "Geography":                       details["geography"],
        "Shares":                          details["shares"],
        "Cost Basis ($)":                  round(details["cost_basis"], 2),
        "Current Price ($)":               round(current_price, 2),
        "Market Value ($)":                round(market_value, 2),
        "Unrealized Profit and Loss ($)":  round(unrealized_pnl, 2),
        "Unrealized Profit and Loss (%)":  round(pnl_pct, 2),
    })

positions_df = pd.DataFrame(positions)
positions_df

,Ticker,Sector,Geography,Shares,Cost Basis ($),Current Price ($),Market Value ($),Unrealized Profit and Loss ($),Unrealized Profit and Loss (%)
0,AAPL,Technology,North America,50,155.0,291.58,14579.00,6829.00,88.12
1,MSFT,Technology,North America,30,280.0,397.36,11920.80,3520.80,41.91
2,JPM,Financials,North America,40,145.0,309.14,12365.60,6565.60,113.20
3,JNJ,Healthcare,North America,25,160.0,238.49,5962.25,1962.25,49.06
4,XOM,Energy,North America,35,105.0,150.62,5271.70,1596.70,43.45
5,ASML,Technology,Europe,15,620.0,1734.19,26012.85,16712.85,179.71
6,NVO,Healthcare,Europe,20,100.0,42.81,856.20,-1143.80,-57.19
7,TM,Consumer,Asia Pacific,20,185.0,172.03,3440.60,-259.40,-7.01


In [6]:
exceptions = nav_df[nav_df["Exception Flag"] == True].copy().reset_index()
exceptions  = exceptions[["Date", "Net Asset Value ($)", "Daily NAV Return (%)"]]
exceptions["Severity"] = exceptions["Daily NAV Return (%)"].abs().apply(
    lambda x: "HIGH" if x > 5 else "MEDIUM"
)

if exceptions.empty:
    print("No exception days in the review period.")
else:
    print(f"{len(exceptions)} exception day(s) flagged:")
    print(exceptions.to_string(index=False))

No exception days in the review period.


In [7]:
NAVY   = "FF1F3864"
WHITE  = "FFFFFFFF"
LGRAY  = "FFF2F2F2"
AMBER  = "FFFFC000"
RED_BG = "FFFFC7CE"
RED_FG = "FF9C0006"
GRN_BG = "FFC6EFCE"
GRN_FG = "FF375623"

def hfill(h): return PatternFill("solid", fgColor=h)
def tborder():
    s = Side(style="thin", color="FFD9D9D9")
    return Border(left=s, right=s, top=s, bottom=s)
def hdr(ws, r, c, val):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(name="Arial", bold=True, color=WHITE, size=9)
    cell.fill      = hfill(NAVY)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border    = tborder()
def dc(ws, r, c, val, fmt=None, bg=None, fg="FF000000", align="right", bold=False):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(name="Arial", size=9, color=fg, bold=bold)
    cell.fill      = hfill(bg) if bg else hfill(LGRAY if r % 2 == 0 else WHITE)
    cell.alignment = Alignment(horizontal=align, vertical="center")
    cell.border    = tborder()
    if fmt: cell.number_format = fmt
def widths(ws, w):
    for i, v in enumerate(w, 1): ws.column_dimensions[get_column_letter(i)].width = v

wb = openpyxl.Workbook()
wb.remove(wb.active)

# Sheet 1: NAV History
ws1 = wb.create_sheet("NAV History")
ws1.sheet_view.showGridLines = False
ws1.freeze_panes = "A3"
ws1.merge_cells("A1:G1")
c = ws1["A1"]
c.value = "Portfolio Net Asset Value History"
c.font  = Font(name="Arial", bold=True, size=12, color=WHITE)
c.fill  = hfill(NAVY)
c.alignment = Alignment(horizontal="center", vertical="center")
ws1.row_dimensions[1].height = 24

for col, h in enumerate(["Date","Gross Asset Value ($)","Daily Expense Accrual ($)",
                          "Net Asset Value ($)","Unit Price ($)",
                          "Daily NAV Return (%)","Exception Flag"], 1):
    hdr(ws1, 2, col, h)
ws1.row_dimensions[2].height = 30

for ri, (date, row) in enumerate(nav_df.iterrows(), 3):
    ws1.row_dimensions[ri].height = 15
    is_exc = row["Exception Flag"]
    bg = RED_BG if is_exc else (LGRAY if ri % 2 == 0 else WHITE)
    dc(ws1, ri, 1, date.strftime("%Y-%m-%d"), align="center", bg=bg)
    dc(ws1, ri, 2, row["Gross Asset Value ($)"],     fmt="$#,##0.00", bg=bg)
    dc(ws1, ri, 3, row["Daily Expense Accrual ($)"], fmt="$#,##0.00", bg=bg)
    dc(ws1, ri, 4, row["Net Asset Value ($)"],       fmt="$#,##0.00", bg=bg)
    dc(ws1, ri, 5, row["Unit Price ($)"],            fmt="$#,##0.0000", bg=bg)
    ret = row["Daily NAV Return (%)"]
    dc(ws1, ri, 6, round(ret,4) if isinstance(ret,float) else ret,
       fmt="0.0000%", fg=RED_FG if isinstance(ret,float) and ret<0 else GRN_FG, bg=bg)
    dc(ws1, ri, 7, "YES - REVIEW" if is_exc else "", align="center",
       fg=RED_FG if is_exc else "FF000000", bold=is_exc, bg=bg)

widths(ws1, [14,22,24,20,16,22,16])

# Sheet 2: Positions
ws2 = wb.create_sheet("Positions Snapshot")
ws2.sheet_view.showGridLines = False
ws2.freeze_panes = "A3"
ws2.merge_cells("A1:I1")
c = ws2["A1"]
c.value = "Current Positions Snapshot"
c.font  = Font(name="Arial", bold=True, size=12, color=WHITE)
c.fill  = hfill(NAVY)
c.alignment = Alignment(horizontal="center", vertical="center")
ws2.row_dimensions[1].height = 24

for col, h in enumerate(["Ticker","Sector","Geography","Shares","Cost Basis ($)",
                          "Current Price ($)","Market Value ($)",
                          "Unrealized Profit and Loss ($)","Unrealized Profit and Loss (%)"], 1):
    hdr(ws2, 2, col, h)
ws2.row_dimensions[2].height = 30

for ri, (_, row) in enumerate(positions_df.iterrows(), 3):
    ws2.row_dimensions[ri].height = 15
    pv  = row["Unrealized Profit and Loss ($)"]
    pbg = GRN_BG if pv >= 0 else RED_BG
    pfg = GRN_FG if pv >= 0 else RED_FG
    dc(ws2, ri, 1, row["Ticker"],            align="center")
    dc(ws2, ri, 2, row["Sector"],            align="left")
    dc(ws2, ri, 3, row["Geography"],         align="left")
    dc(ws2, ri, 4, row["Shares"],            fmt="#,##0")
    dc(ws2, ri, 5, row["Cost Basis ($)"],    fmt="$#,##0.00")
    dc(ws2, ri, 6, row["Current Price ($)"], fmt="$#,##0.00")
    dc(ws2, ri, 7, row["Market Value ($)"],  fmt="$#,##0.00")
    dc(ws2, ri, 8, pv,                       fmt="$#,##0.00", fg=pfg, bg=pbg)
    dc(ws2, ri, 9, row["Unrealized Profit and Loss (%)"], fmt="0.00%", fg=pfg, bg=pbg)

widths(ws2, [10,16,16,8,14,16,16,28,28])

# Sheet 3: Exception Log
ws3 = wb.create_sheet("Exception Log")
ws3.sheet_view.showGridLines = False
ws3.freeze_panes = "A3"
ws3.merge_cells("A1:D1")
c = ws3["A1"]
c.value = "Net Asset Value Exception Log"
c.font  = Font(name="Arial", bold=True, size=12, color=WHITE)
c.fill  = hfill(NAVY)
c.alignment = Alignment(horizontal="center", vertical="center")
ws3.row_dimensions[1].height = 24

for col, h in enumerate(["Date","Net Asset Value ($)","Daily NAV Return (%)","Severity"], 1):
    hdr(ws3, 2, col, h)
ws3.row_dimensions[2].height = 30

if exceptions.empty:
    ws3.merge_cells("A3:D3")
    c = ws3["A3"]
    c.value = "No exceptions recorded in the review period."
    c.font  = Font(name="Arial", size=9, italic=True, color="FF404040")
    c.alignment = Alignment(horizontal="center", vertical="center")
else:
    for ri, (_, row) in enumerate(exceptions.iterrows(), 3):
        ws3.row_dimensions[ri].height = 15
        sbg = RED_BG if row["Severity"] == "HIGH" else AMBER
        sfg = RED_FG if row["Severity"] == "HIGH" else "FF633806"
        dc(ws3, ri, 1, row["Date"].strftime("%Y-%m-%d"), align="center")
        dc(ws3, ri, 2, row["Net Asset Value ($)"],       fmt="$#,##0.00")
        dc(ws3, ri, 3, round(row["Daily NAV Return (%)"],4), fmt="0.0000%")
        dc(ws3, ri, 4, row["Severity"], align="center", fg=sfg, bg=sbg, bold=True)

widths(ws3, [14,22,22,12])

# Save
output_dir  = r"C:\Users\Anagha Athreyas\OneDrive\Desktop\GitHub Content\NAV Tracker\output"
os.makedirs(output_dir, exist_ok=True)
filename    = f"NAV_Report_{datetime.today().strftime('%Y%m%d')}.xlsx"
output_path = os.path.join(output_dir, filename)
wb.save(output_path)
print(f"Report saved to: {output_path}")

Report saved to: C:\Users\Anagha Athreyas\OneDrive\Desktop\GitHub Content\NAV Tracker\output\NAV_Report_20260611.xlsx
